# LeetCode 4103 — Remove Elements by Type and Color (Medium)

Bài toán LeetCode:

```python
class Solution:
    def canRemoveAll(self, types: List[int], colors: List[int], groupSize: List[int]) -> bool:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4103 (Biweekly Contest 195 — câu 2).

Đây là bài "xoá khối rồi dồn lại" — giống trò chơi xếp kẹo: xoá một nhóm thì hai phía dồn sát vào nhau và có thể tạo ra nhóm mới. Kỹ thuật chính là **ngăn xếp (stack) lưu các khối đang chờ**, cùng với một lập luận rất đẹp: **thứ tự xoá không quan trọng**.

---

## Sau notebook này, bạn sẽ nắm được

1. Đề bài yêu cầu gì, và vì sao "loại + màu" có thể gộp thành một **khóa (key)** duy nhất.
2. Cách vét cạn mọi thứ tự xoá (đúng nhưng bùng nổ số trạng thái).
3. Thí nghiệm cho thấy **mọi thứ tự xoá đều dẫn tới cùng một kết quả cuối** — và vì sao.
4. Cách mô phỏng tham lam: cứ thấy nhóm đủ lớn là xoá.
5. Kỹ thuật **stack lưu khối `[key, số lượng]`** cho lời giải `O(n)`.
6. Các lỗi hay gặp: quên so màu, đếm theo khối ban đầu mà quên hiệu ứng dây chuyền, ...
7. Mẹo nhận diện họ bài "xoá k phần tử liền nhau giống nhau" và bài tập vận dụng.

# 1. Đề bài yêu cầu gì?

Có `n` phần tử xếp thành một hàng. Phần tử thứ `i` có:

- **loại** `types[i]` (một số từ `0` tới `m - 1`),
- **màu** `colors[i]`.

Mảng `groupSize` dài `m`: `groupSize[t]` là **kích thước nhóm bắt buộc** của loại `t`.

**Một thao tác:** chọn đúng `groupSize[t]` phần tử **liên tiếp**, tất cả **cùng loại `t`** và **cùng màu**, rồi xoá chúng. Các phần tử còn lại **dồn sát vào nhau** (khoảng trống biến mất).

Trả về `True` nếu có thể xoá **hết** mọi phần tử bằng một dãy thao tác hợp lệ, ngược lại `False`.

---

## Ví dụ 1

```python
types = [0, 1, 1, 0], colors = [5, 3, 3, 5], groupSize = [2, 2]   ->   True
```

- Hai phần tử ở giữa cùng loại `1`, cùng màu `3`, và `groupSize[1] = 2` → xoá.
- Còn lại hai phần tử loại `0`, màu `5`, giờ đã **dồn sát nhau**, `groupSize[0] = 2` → xoá. Hết!

## Ví dụ 2

```python
types = [0, 1, 0], colors = [9, 9, 9], groupSize = [3, 2]   ->   False
```

Cùng màu nhưng khác loại. Loại `0` cần 3 phần tử liền nhau, loại `1` cần 2 → không có thao tác nào làm được.

## Ví dụ 3

```python
types = [2, 0, 1, 2], colors = [7, 5, 4, 7], groupSize = [3, 1, 2]   ->   False
```

- Phần tử chỉ số `2` (loại `1`, màu `4`) có `groupSize[1] = 1` → xoá một mình được.
- Còn lại `types = [2, 0, 2]`, `colors = [7, 5, 7]`. Loại `0` cần 3, loại `2` cần 2 nhưng hai phần tử loại `2` bị phần tử loại `0` chặn ở giữa → kẹt.

---

## Ràng buộc

```text
1 <= n = types.length = colors.length <= 10^5
1 <= m = groupSize.length <= 10^5
0 <= types[i] <= m - 1
1 <= colors[i] <= 10^5
1 <= groupSize[t] <= n
```

# 2. Kiến thức nền tảng

## 2.1. Gộp (loại, màu) thành một khóa bằng `tuple`

Hai phần tử chỉ có thể nằm chung nhóm khi **cùng loại và cùng màu**. Vậy ta coi cặp `(type, color)` như **một khóa duy nhất**. Trong Python, `tuple` so sánh bằng nhau khi từng thành phần bằng nhau.

`zip(types, colors)` ghép hai mảng thành dãy các cặp.

In [1]:
types = [0, 1, 1, 0]
colors = [5, 3, 3, 5]

keys = list(zip(types, colors))
print(keys)
print(keys[1] == keys[2])     # (1, 3) == (1, 3) -> True
print(keys[0] == keys[1])     # (0, 5) == (1, 3) -> False
print((0, 9) == (1, 9))       # cùng màu, khác loại -> False

[(0, 5), (1, 3), (1, 3), (0, 5)]
True
False
False


Kích thước nhóm cần cho khóa `(t, c)` là `groupSize[t]` — chỉ phụ thuộc **loại**, không phụ thuộc màu.

## 2.2. `list` dùng như một ngăn xếp (stack)

Stack là cấu trúc "vào sau, ra trước" (LIFO — Last In, First Out), giống chồng đĩa: chỉ thao tác ở **đỉnh**.

| Thao tác | Python | Ý nghĩa |
|---|---|---|
| đẩy vào | `st.append(x)` | đặt `x` lên đỉnh |
| lấy ra | `st.pop()` | bỏ phần tử ở đỉnh (và trả về nó) |
| xem đỉnh | `st[-1]` | đọc phần tử ở đỉnh, không bỏ |
| rỗng? | `not st` | `True` nếu stack rỗng |

Cả ba thao tác đều `O(1)`.

In [2]:
st = []
st.append("A")
st.append("B")
st.append("C")
print("stack:", st, "| đỉnh:", st[-1])

top = st.pop()
print("vừa lấy ra:", top, "| stack còn:", st)

print("rỗng?", not st)
st.pop(); st.pop()
print("rỗng?", not st)

stack: ['A', 'B', 'C'] | đỉnh: C
vừa lấy ra: C | stack còn: ['A', 'B']
rỗng? False
rỗng? True


## 2.3. Phần tử của stack có thể là một list để **sửa tại chỗ**

Ta sẽ lưu mỗi "khối" là `[type, color, count]`. Vì là `list` (không phải `tuple`), ta có thể tăng `count` ngay trên đỉnh: `st[-1][2] += 1`.

In [3]:
st = [[0, 5, 1]]          # khối: loại 0, màu 5, đang có 1 phần tử
st[-1][2] += 1            # thêm một phần tử cùng khóa vào khối đỉnh
print(st)

[[0, 5, 2]]


## 2.4. Xoá một đoạn liên tiếp trong list: `del a[i:j]`

`del a[i:j]` xoá các phần tử từ chỉ số `i` tới trước `j`, rồi phần sau **tự dồn lên**. Đây chính là "dồn lại" trong đề. Lưu ý: thao tác này tốn `O(n)` vì phải dịch các phần tử phía sau.

In [4]:
a = ["x", "B", "B", "y"]
del a[1:3]
print(a)          # ['x', 'y'] - hai phần tử hai bên đã dồn sát nhau

['x', 'y']


# 3. Cách 1 — Vét cạn mọi thứ tự xoá

Cách "an toàn tuyệt đối": ở mỗi trạng thái, thử **mọi** thao tác có thể làm, đệ quy tiếp. Nếu có một nhánh xoá được hết → `True`.

- **Trạng thái** = dãy khóa hiện tại, lưu dạng `tuple` để có thể dùng làm key của bộ nhớ đệm (memo).
- Thao tác hợp lệ tại vị trí `i`: `groupSize[t]` phần tử bắt đầu từ `i` đều bằng khóa `(t, c)`.

`functools.lru_cache` giúp nhớ kết quả các trạng thái đã xét để không xét lại.

In [5]:
from functools import lru_cache
from typing import List


def can_remove_exhaustive(types: List[int], colors: List[int], groupSize: List[int]) -> bool:
    @lru_cache(maxsize=None)
    def can(state: tuple) -> bool:
        if not state:
            return True                       # đã xoá hết
        n = len(state)
        for i in range(n):
            t, c = state[i]
            need = groupSize[t]
            if i + need <= n and all(state[k] == (t, c) for k in range(i, i + need)):
                if can(state[:i] + state[i + need:]):   # xoá rồi đệ quy
                    return True
        return False

    return can(tuple(zip(types, colors)))


print(can_remove_exhaustive([0, 1, 1, 0], [5, 3, 3, 5], [2, 2]))          # True
print(can_remove_exhaustive([0, 1, 0], [9, 9, 9], [3, 2]))                # False
print(can_remove_exhaustive([2, 0, 1, 2], [7, 5, 4, 7], [3, 1, 2]))       # False

True
False
False


## Vì sao không dùng được?

Số trạng thái có thể sinh ra tăng rất nhanh khi có nhiều lựa chọn xoá. Cell dưới đếm số trạng thái khác nhau mà cách vét cạn phải xét với dãy toàn phần tử `groupSize = 1` (mỗi phần tử tự xoá được — trường hợp nhiều lựa chọn nhất).

In [6]:
def count_states(keys: tuple, groupSize: List[int]) -> int:
    seen = set()
    stack = [keys]
    while stack:
        state = stack.pop()
        if state in seen:
            continue
        seen.add(state)
        n = len(state)
        for i in range(n):
            t, c = state[i]
            need = groupSize[t]
            if i + need <= n and all(state[k] == (t, c) for k in range(i, i + need)):
                stack.append(state[:i] + state[i + need:])
    return len(seen)


for n in [4, 8, 12, 16]:
    keys = tuple((0, i) for i in range(n))     # n phần tử khác màu, loại 0, groupSize[0] = 1
    print(f"n = {n:>2}: {count_states(keys, [1]):>6,} trạng thái")

n =  4:     16 trạng thái
n =  8:    256 trạng thái
n = 12:  4,096 trạng thái


n = 16: 65,536 trạng thái


Mỗi tập con phần tử còn lại là một trạng thái → `2^n` trạng thái. Với `n = 10^5` thì không tưởng.

Câu hỏi then chốt:

> **Có thật sự cần thử mọi thứ tự xoá không? Hay thứ tự nào cũng như nhau?**

---

# 4. Thí nghiệm: thứ tự xoá có quan trọng không?

Ta viết hàm tìm **tất cả trạng thái cuối** (trạng thái không còn thao tác nào) có thể đạt được từ input. Nếu thứ tự xoá quan trọng, ta sẽ thấy **nhiều** trạng thái cuối khác nhau.

In [7]:
def final_states(types: List[int], colors: List[int], groupSize: List[int]) -> set:
    finals = set()
    seen = set()
    stack = [tuple(zip(types, colors))]
    while stack:
        state = stack.pop()
        if state in seen:
            continue
        seen.add(state)
        moved = False
        n = len(state)
        for i in range(n):
            t, c = state[i]
            need = groupSize[t]
            if i + need <= n and all(state[k] == (t, c) for k in range(i, i + need)):
                stack.append(state[:i] + state[i + need:])
                moved = True
        if not moved:
            finals.add(state)                 # không xoá thêm được nữa
    return finals


print(final_states([0, 1, 1, 0], [5, 3, 3, 5], [2, 2]))
print(final_states([2, 0, 1, 2], [7, 5, 4, 7], [3, 1, 2]))
print(final_states([0, 0, 0, 1, 1], [1, 1, 1, 2, 2], [2, 1]))

{()}
{((2, 7), (0, 5), (2, 7))}
{((0, 1),)}


Mỗi lần chỉ có **một** trạng thái cuối duy nhất. Thử trên hàng nghìn input ngẫu nhiên:

In [8]:
import random

random.seed(4103)
max_finals = 0
for _ in range(3000):
    m = random.randint(1, 3)
    groupSize = [random.randint(1, 3) for _ in range(m)]
    n = random.randint(1, 9)
    types = [random.randrange(m) for _ in range(n)]
    colors = [random.randint(1, 2) for _ in range(n)]
    max_finals = max(max_finals, len(final_states(types, colors, groupSize)))

print("Số trạng thái cuối khác nhau nhiều nhất từng gặp:", max_finals)

Số trạng thái cuối khác nhau nhiều nhất từng gặp: 1


## Vì sao thứ tự không quan trọng?

Giả sử đang có **hai** thao tác hợp lệ khác nhau, gọi là `X` và `Y`. Chỉ có hai khả năng:

**Trường hợp 1 — `X` và `Y` không chồng lên nhau.** Làm `X` không phá hỏng `Y` (các phần tử của `Y` vẫn liền nhau, cùng khóa) và ngược lại. Làm `X` rồi `Y` hay `Y` rồi `X` cho **cùng một kết quả**.

```text
... [X X] ... [Y Y Y] ...    -> xoá cái nào trước cũng được, cuối cùng cả hai đều biến mất
```

**Trường hợp 2 — `X` và `Y` chồng lên nhau.** Khi đó chúng có chung phần tử nên phải **cùng khóa**, tức là cùng nằm trong một dải dài các phần tử giống hệt nhau. Xoá `g` phần tử liên tiếp **bất kỳ** trong một dải giống hệt nhau đều để lại **đúng cùng một dãy** (dải chỉ ngắn đi `g`).

```text
dải: A A A A A  (g = 2)
xoá 2 cái đầu: A A A        xoá 2 cái giữa: A A A    -> giống hệt nhau
```

Vậy hai lựa chọn khác nhau luôn có thể "hội tụ" về cùng một trạng thái. Thêm nữa, mỗi thao tác làm dãy ngắn đi nên quá trình chắc chắn dừng. Trong lý thuyết, hai tính chất này (luôn dừng + hai nhánh luôn hội tụ được) đảm bảo **trạng thái cuối là duy nhất** — đây là nội dung của *bổ đề Newman (Newman's lemma)*.

> **Hệ quả cực mạnh:** không cần thử mọi thứ tự. Cứ thấy nhóm nào xoá được là xoá — kết quả cuối vẫn như nhau. Xoá hết được hay không chỉ phụ thuộc trạng thái cuối duy nhất đó có rỗng hay không.

# 5. Cách 2 — Mô phỏng tham lam

Nhờ kết luận trên, ta làm đơn giản:

1. Duyệt dãy, tìm **dải** (run) đầu tiên gồm các phần tử cùng khóa có độ dài `>= groupSize[t]`.
2. Xoá `groupSize[t]` phần tử của dải đó.
3. Lặp lại từ đầu cho tới khi không xoá được nữa.
4. Trả về `True` nếu dãy rỗng.

Ví dụ dây chuyền: `types = [0, 1, 2, 2, 1, 0]`, `colors = [1, 2, 3, 3, 2, 1]`, `groupSize = [2, 2, 2]`:

```text
(0,1) (1,2) (2,3) (2,3) (1,2) (0,1)
             ^^^^^^^^^^^  xoá
(0,1) (1,2) (1,2) (0,1)
       ^^^^^^^^^^^        xoá (vừa được dồn sát nhau)
(0,1) (0,1)
^^^^^^^^^^^               xoá
[]                        -> True
```

In [9]:
def can_remove_simulation(types: List[int], colors: List[int], groupSize: List[int],
                          verbose: bool = False) -> bool:
    arr = list(zip(types, colors))
    changed = True
    while changed:
        changed = False
        i = 0
        while i < len(arr):
            j = i
            while j < len(arr) and arr[j] == arr[i]:   # tìm hết dải bắt đầu từ i
                j += 1
            need = groupSize[arr[i][0]]
            if j - i >= need:                           # dải đủ dài -> xoá need phần tử
                if verbose:
                    print(f"xoá {need} x {arr[i]} tại vị trí {i}: {arr}")
                del arr[i:i + need]
                changed = True
                break                                   # dãy đã đổi, quét lại từ đầu
            i = j
    if verbose:
        print("trạng thái cuối:", arr)
    return not arr


print(can_remove_simulation([0, 1, 2, 2, 1, 0], [1, 2, 3, 3, 2, 1], [2, 2, 2], verbose=True))
print()
print(can_remove_simulation([2, 0, 1, 2], [7, 5, 4, 7], [3, 1, 2], verbose=True))

xoá 2 x (2, 3) tại vị trí 2: [(0, 1), (1, 2), (2, 3), (2, 3), (1, 2), (0, 1)]
xoá 2 x (1, 2) tại vị trí 1: [(0, 1), (1, 2), (1, 2), (0, 1)]
xoá 2 x (0, 1) tại vị trí 0: [(0, 1), (0, 1)]
trạng thái cuối: []
True

xoá 1 x (1, 4) tại vị trí 2: [(2, 7), (0, 5), (1, 4), (2, 7)]
trạng thái cuối: [(2, 7), (0, 5), (2, 7)]
False


## Độ phức tạp của mô phỏng

Mỗi lần xoá ta quét lại từ đầu (`O(n)`) và `del` cũng tốn `O(n)`. Có thể có tới `O(n)` lần xoá → **`O(n²)`**. Với dây chuyền lồng nhau rất sâu, `n = 10^5` sẽ mất hàng chục tỷ bước.

Điểm lãng phí: sau mỗi lần xoá, ta quét lại **từ đầu dãy**, trong khi chỉ có chỗ vừa dồn lại là thay đổi.

---

# 6. Bước đột phá: đọc từ trái sang phải, giữ các khối "đang chờ" trong stack

Hãy tưởng tượng ta đọc các phần tử **từ trái sang phải** và giữ một **chồng khối** đã đọc nhưng chưa xoá được. Mỗi khối là `[type, color, count]` — `count` phần tử liền nhau cùng khóa.

Khi đọc phần tử mới có khóa `(t, c)`:

1. Nếu khối trên đỉnh **cùng khóa** → tăng `count` của khối đỉnh.
2. Ngược lại → đẩy một khối mới `[t, c, 1]`.
3. Nếu khối đỉnh đạt đúng `groupSize[t]` → **xoá ngay**: `pop` khối đó.

Điều kỳ diệu nằm ở bước 3: sau khi `pop`, khối **bên dưới** lộ ra trở thành đỉnh. Nếu phần tử tiếp theo cùng khóa với nó, chúng sẽ **nối tiếp** nhau — đúng như việc "dồn sát lại" sau khi xoá!

Đọc hết dãy, nếu stack rỗng → xoá được hết.

Đây là cùng ý tưởng với bài kinh điển *Remove All Adjacent Duplicates in String II* (LeetCode 1209), chỉ khác là "khóa" là cặp (loại, màu) và kích thước nhóm phụ thuộc loại.

## Mô phỏng tay với ví dụ dây chuyền

`keys = (0,1) (1,2) (2,3) (2,3) (1,2) (0,1)`, `groupSize = [2, 2, 2]`

| Bước | Đọc | Đỉnh trước | Hành động | Stack sau |
|---:|---|---|---|---|
| 1 | (0,1) | rỗng | đẩy khối mới | `[[0,1,1]]` |
| 2 | (1,2) | (0,1) | khác khóa → đẩy | `[[0,1,1], [1,2,1]]` |
| 3 | (2,3) | (1,2) | khác khóa → đẩy | `[[0,1,1], [1,2,1], [2,3,1]]` |
| 4 | (2,3) | (2,3) | cùng khóa → count = 2 = groupSize[2] → **pop** | `[[0,1,1], [1,2,1]]` |
| 5 | (1,2) | (1,2) | cùng khóa → count = 2 → **pop** | `[[0,1,1]]` |
| 6 | (0,1) | (0,1) | cùng khóa → count = 2 → **pop** | `[]` |

Stack rỗng → `True`.

Để ý bước 5: khối `(1,2)` được đẩy từ bước 2, bị "che" bởi khối `(2,3)`. Khi `(2,3)` bị xoá ở bước 4, nó lộ ra và nối với phần tử `(1,2)` mới đọc — chính là hiệu ứng dồn lại.

# 7. Cách 3 — Stack khối: lời giải O(n)

In [10]:
def can_remove_stack(types: List[int], colors: List[int], groupSize: List[int]) -> bool:
    st = []                                    # mỗi phần tử: [type, color, count]
    for t, c in zip(types, colors):
        if st and st[-1][0] == t and st[-1][1] == c:
            st[-1][2] += 1                     # nối vào khối đỉnh
        else:
            st.append([t, c, 1])               # mở khối mới
        if st[-1][2] == groupSize[t]:
            st.pop()                           # đủ nhóm -> xoá ngay
    return not st


print(can_remove_stack([0, 1, 1, 0], [5, 3, 3, 5], [2, 2]))               # True
print(can_remove_stack([0, 1, 0], [9, 9, 9], [3, 2]))                     # False
print(can_remove_stack([2, 0, 1, 2], [7, 5, 4, 7], [3, 1, 2]))            # False
print(can_remove_stack([0, 1, 2, 2, 1, 0], [1, 2, 3, 3, 2, 1], [2, 2, 2]))  # True

True
False
False
True


## Trace bằng `print`

Hàm dưới in trạng thái stack sau từng phần tử. Ví dụ thứ hai có một dải **dài hơn** kích thước nhóm (`groupSize[0] = 2` nhưng có 3 phần tử `(0,1)` liền nhau) để thấy phần dư bị kẹt lại.

In [11]:
def trace_stack(types: List[int], colors: List[int], groupSize: List[int]) -> bool:
    st = []
    for i, (t, c) in enumerate(zip(types, colors)):
        if st and st[-1][0] == t and st[-1][1] == c:
            st[-1][2] += 1
            action = f"nối vào đỉnh, count = {st[-1][2]}"
        else:
            st.append([t, c, 1])
            action = "mở khối mới"
        if st[-1][2] == groupSize[t]:
            st.pop()
            action += f" -> đủ {groupSize[t]} -> XOÁ khối"
        print(f"i = {i}: đọc ({t},{c}) | {action:40} | stack = {st}")
    print("Kết quả:", not st)
    return not st


trace_stack([0, 1, 2, 2, 1, 0], [1, 2, 3, 3, 2, 1], [2, 2, 2])
print()
trace_stack([0, 0, 0, 1], [1, 1, 1, 4], [2, 1])

i = 0: đọc (0,1) | mở khối mới                              | stack = [[0, 1, 1]]
i = 1: đọc (1,2) | mở khối mới                              | stack = [[0, 1, 1], [1, 2, 1]]
i = 2: đọc (2,3) | mở khối mới                              | stack = [[0, 1, 1], [1, 2, 1], [2, 3, 1]]
i = 3: đọc (2,3) | nối vào đỉnh, count = 2 -> đủ 2 -> XOÁ khối | stack = [[0, 1, 1], [1, 2, 1]]
i = 4: đọc (1,2) | nối vào đỉnh, count = 2 -> đủ 2 -> XOÁ khối | stack = [[0, 1, 1]]
i = 5: đọc (0,1) | nối vào đỉnh, count = 2 -> đủ 2 -> XOÁ khối | stack = []
Kết quả: True

i = 0: đọc (0,1) | mở khối mới                              | stack = [[0, 1, 1]]
i = 1: đọc (0,1) | nối vào đỉnh, count = 2 -> đủ 2 -> XOÁ khối | stack = []
i = 2: đọc (0,1) | mở khối mới                              | stack = [[0, 1, 1]]
i = 3: đọc (1,4) | mở khối mới -> đủ 1 -> XOÁ khối          | stack = [[0, 1, 1]]
Kết quả: False


False

# 8. Phiên bản đúng format LeetCode

In [12]:
from typing import List


class Solution:
    def canRemoveAll(self, types: List[int], colors: List[int], groupSize: List[int]) -> bool:
        st = []                                # [type, color, count]
        for t, c in zip(types, colors):
            if st and st[-1][0] == t and st[-1][1] == c:
                st[-1][2] += 1
            else:
                st.append([t, c, 1])
            if st[-1][2] == groupSize[t]:
                st.pop()
        return not st

In [13]:
sol = Solution()

tests = [
    ([0, 1, 1, 0], [5, 3, 3, 5], [2, 2], True, "ví dụ 1"),
    ([0, 1, 0], [9, 9, 9], [3, 2], False, "ví dụ 2"),
    ([2, 0, 1, 2], [7, 5, 4, 7], [3, 1, 2], False, "ví dụ 3"),
    ([0], [1], [1], True, "một phần tử, nhóm 1"),
    ([0], [1], [2], False, "một phần tử, nhóm 2"),
    ([0, 0, 0], [1, 1, 1], [2], False, "dải 3, nhóm 2 -> dư 1"),
    ([0, 0, 0, 0], [1, 1, 1, 1], [2], True, "dải 4, nhóm 2"),
    ([0, 0, 1, 1], [1, 2, 1, 2], [1, 1], True, "nhóm 1: luôn xoá được"),
    ([0, 1, 0], [1, 2, 1], [2, 1], True, "dồn lại tạo nhóm mới"),
    ([0, 0], [1, 2], [2], False, "cùng loại khác màu"),
    ([0, 1, 2, 2, 1, 0], [1, 2, 3, 3, 2, 1], [2, 2, 2], True, "dây chuyền lồng nhau"),
]

for types, colors, groupSize, expected, name in tests:
    result = sol.canRemoveAll(types, colors, groupSize)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:26} -> {result}")

PASS | ví dụ 1                    -> True
PASS | ví dụ 2                    -> False
PASS | ví dụ 3                    -> False
PASS | một phần tử, nhóm 1        -> True
PASS | một phần tử, nhóm 2        -> False
PASS | dải 3, nhóm 2 -> dư 1      -> False
PASS | dải 4, nhóm 2              -> True
PASS | nhóm 1: luôn xoá được      -> True
PASS | dồn lại tạo nhóm mới       -> True
PASS | cùng loại khác màu         -> False
PASS | dây chuyền lồng nhau       -> True


# 9. Vì sao stack đúng?

**Bất biến của stack:** sau khi đọc xong `k` phần tử đầu tiên, stack mô tả **trạng thái cuối duy nhất** của đoạn đầu dài `k` đó (nếu chỉ xét riêng đoạn này):

- hai khối kề nhau trong stack luôn **khác khóa** (vì cùng khóa thì đã được nối vào nhau thay vì mở khối mới);
- mọi khối đều có `count < groupSize[type]` (vì đạt đủ là bị `pop` ngay).

Nghĩa là trong đoạn đầu, **không còn thao tác nào** làm được nữa.

Khi đọc thêm một phần tử, nó chỉ có thể tương tác với khối **cuối cùng** (đỉnh stack) — phần tử mới nằm sát khối đó. Nếu nối vào làm khối đủ lớn thì xoá; sau khi xoá, khối bên dưới trở thành phần tử cuối, sẵn sàng nối với phần tử kế tiếp. Không có cơ hội xoá nào bị bỏ sót.

Stack thực chất là **một thứ tự xoá cụ thể** ("thấy đủ là xoá ngay, ưu tiên phía phải"). Theo mục 4, mọi thứ tự xoá cho cùng trạng thái cuối, nên stack rỗng **khi và chỉ khi** có thể xoá hết.

---

## Trực giác

> Stack giống một hàng đợi "chờ ghép đôi": mỗi khối nằm chờ cho tới khi có đủ đồng đội. Khi một khối được xoá, khối nằm dưới nó "thức dậy" và tiếp tục chờ.

# 10. Độ phức tạp

Gọi `n = len(types)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---:|---:|
| 1. Vét cạn | thử mọi thứ tự xoá + memo | mũ (tới `2^n` trạng thái) | mũ |
| 2. Mô phỏng tham lam | tìm dải đủ dài, xoá, quét lại | `O(n²)` | `O(n)` |
| 3. Stack khối | đọc một lượt, nối / xoá ở đỉnh | `O(n)` | `O(n)` |

Ở cách 3, mỗi phần tử gây ra **tối đa một** lần `append` hoặc một lần tăng `count`, và mỗi khối bị `pop` **tối đa một** lần → tổng `O(n)`.

So sánh tốc độ thực tế giữa cách 2 và cách 3 trên một dây chuyền lồng nhau sâu:

In [14]:
import time

def nested_chain(depth: int):
    # (0,1) (0,2) ... (0,depth) (0,depth) ... (0,2) (0,1), groupSize = [2]
    colors = list(range(1, depth + 1)) + list(range(depth, 0, -1))
    return [0] * len(colors), colors, [2]

for depth in [500, 1000, 2000]:
    types, colors, groupSize = nested_chain(depth)
    t0 = time.perf_counter(); r2 = can_remove_simulation(types, colors, groupSize); t2 = time.perf_counter() - t0
    t0 = time.perf_counter(); r3 = can_remove_stack(types, colors, groupSize); t3 = time.perf_counter() - t0
    print(f"n = {2 * depth:>4}: mô phỏng {t2:.3f}s | stack {t3:.5f}s | kết quả {r2}, {r3}")

n = 1000: mô phỏng 0.017s | stack 0.00013s | kết quả True, True
n = 2000: mô phỏng 0.070s | stack 0.00046s | kết quả True, True


n = 4000: mô phỏng 0.331s | stack 0.00082s | kết quả True, True


Thời gian của mô phỏng tăng khoảng **3–4 lần** mỗi khi `n` gấp đôi (đặc trưng của `O(n²)`; về lý thuyết là gấp 4), còn stack gần như không đáng kể.

# 11. Đối chiếu ba cách trên dữ liệu ngẫu nhiên

In [15]:
random.seed(2026)
count_true = 0
for _ in range(3000):
    m = random.randint(1, 3)
    groupSize = [random.randint(1, 3) for _ in range(m)]
    n = random.randint(1, 9)
    types = [random.randrange(m) for _ in range(n)]
    colors = [random.randint(1, 2) for _ in range(n)]

    expected = can_remove_exhaustive(types, colors, groupSize)
    assert can_remove_simulation(types, colors, groupSize) == expected
    assert can_remove_stack(types, colors, groupSize) == expected
    assert sol.canRemoveAll(types, colors, groupSize) == expected
    count_true += expected

print(f"3000 bộ test ngẫu nhiên khớp nhau ({count_true} bộ True, {3000 - count_true} bộ False)")

3000 bộ test ngẫu nhiên khớp nhau (753 bộ True, 2247 bộ False)


# 12. Những lỗi dễ mắc

## Lỗi 1 — Chỉ so sánh loại, quên so màu

Code sai:

```python
if st and st[-1][0] == t:      # thiếu điều kiện cùng màu
```

Hai phần tử cùng loại nhưng khác màu **không** được ghép nhóm.

In [16]:
def wrong_ignore_color(types, colors, groupSize):
    st = []
    for t, c in zip(types, colors):
        if st and st[-1][0] == t:              # SAI: quên so màu
            st[-1][2] += 1
        else:
            st.append([t, c, 1])
        if st[-1][2] == groupSize[t]:
            st.pop()
    return not st


print("Sai :", wrong_ignore_color([0, 0], [1, 2], [2]))
print("Đúng:", sol.canRemoveAll([0, 0], [1, 2], [2]))

Sai : True
Đúng: False


## Lỗi 2 — Chỉ nhìn các dải ban đầu, quên hiệu ứng dây chuyền

Một ý tưởng "nghe có vẻ đúng": tách dãy ban đầu thành các dải cùng khóa, rồi kiểm tra mỗi dải có độ dài chia hết cho `groupSize` không.

Sai, vì sau khi xoá, các dải **nối lại** với nhau. Ví dụ `(0,1) (1,2) (0,1)` với `groupSize = [2, 1]`: hai dải `(0,1)` đều dài `1` (không chia hết cho `2`), nhưng xoá `(1,2)` xong thì chúng dồn lại thành dải dài `2`.

In [17]:
from itertools import groupby

def wrong_initial_runs(types, colors, groupSize):
    for key, grp in groupby(zip(types, colors)):
        length = len(list(grp))
        if length % groupSize[key[0]] != 0:
            return False
    return True


print("Sai :", wrong_initial_runs([0, 1, 0], [1, 2, 1], [2, 1]))
print("Đúng:", sol.canRemoveAll([0, 1, 0], [1, 2, 1], [2, 1]))

Sai : False
Đúng: True


## Lỗi 3 — Mỗi phần tử đều đẩy một khối mới, không nối vào đỉnh

Nếu luôn `append([t, c, 1])`, `count` không bao giờ vượt quá `1` → chỉ những loại có `groupSize = 1` mới bị xoá. Phải kiểm tra đỉnh trước: cùng khóa thì **tăng count**.

---

## Lỗi 4 — Lấy kích thước nhóm theo màu hoặc theo chỉ số phần tử

Kích thước nhóm là `groupSize[t]` với `t` là **loại**. Viết nhầm `groupSize[c]` hay `groupSize[i]` vẫn có thể chạy được (khi chỉ số không vượt mảng) nhưng cho kết quả sai, rất khó phát hiện.

---

## Lỗi 5 — So sánh `count >= groupSize[t]` rồi trừ, thay vì `pop` ngay khi bằng

Với stack, khối bị xoá **ngay khi** `count == groupSize[t]`, nên `count` không bao giờ vượt quá. Viết `>=` không sai, nhưng nếu kết hợp với việc trừ `count -= groupSize[t]` mà quên `pop` khi `count` về `0`, sẽ để lại khối rỗng `[t, c, 0]` làm stack không bao giờ rỗng.

---

## Lỗi 6 — Dùng mô phỏng `O(n²)` hoặc vét cạn cho `n = 10^5`

Đúng với ví dụ nhỏ nhưng TLE. Hãy nhớ: thấy "xoá rồi dồn lại" → nghĩ ngay tới stack.

# 13. Mẹo nhận diện

Nghĩ tới **stack lưu khối `[key, count]`** khi đề có các dấu hiệu:

1. **Xoá một nhóm phần tử liền nhau giống nhau**, phần còn lại **dồn sát lại**.
2. Việc xoá có thể gây **phản ứng dây chuyền** (hai phía mới gặp nhau tạo nhóm mới).
3. Câu hỏi: kết quả cuối cùng là gì / có xoá hết được không.

Kèm theo đó, hãy tự hỏi: **"thứ tự xoá có quan trọng không?"** Nếu hai thao tác chồng lên nhau luôn cho cùng kết quả (như bài này), ta được phép xoá tham lam.

Các bài cùng họ:

- *Remove All Adjacent Duplicates In String* (LeetCode 1047) — xoá 2 ký tự giống nhau liền kề.
- *Remove All Adjacent Duplicates in String II* (LeetCode 1209) — xoá `k` ký tự giống nhau liền kề.
- *Valid Parentheses* (LeetCode 20) — cặp `(` `)` "triệt tiêu" nhau, cũng là stack.

Cảnh báo: có những bài **thứ tự xoá rất quan trọng** (ví dụ *Zuma Game*, *Remove Boxes*) — khi đó không dùng được tham lam mà phải quy hoạch động / vét cạn.

# 14. Biến thể

## Biến thể 1 — Trả về dãy còn lại sau khi xoá hết mức có thể

Stack chính là "dạng rút gọn" của dãy. Chỉ cần "bung" mỗi khối thành `count` phần tử.

In [18]:
def remaining_after_removal(types: List[int], colors: List[int], groupSize: List[int]) -> List[tuple]:
    st = []
    for t, c in zip(types, colors):
        if st and st[-1][0] == t and st[-1][1] == c:
            st[-1][2] += 1
        else:
            st.append([t, c, 1])
        if st[-1][2] == groupSize[t]:
            st.pop()
    result = []
    for t, c, cnt in st:
        result.extend([(t, c)] * cnt)
    return result


print(remaining_after_removal([2, 0, 1, 2], [7, 5, 4, 7], [3, 1, 2]))
print(remaining_after_removal([0, 0, 0, 1], [1, 1, 1, 4], [2, 1]))

[(2, 7), (0, 5), (2, 7)]
[(0, 1)]


## Biến thể 2 — Xoá `k` ký tự giống nhau liền kề trong chuỗi (LeetCode 1209)

Khóa là chính ký tự, mọi ký tự có chung kích thước nhóm `k`. Kết quả là chuỗi còn lại.

In [19]:
def remove_k_adjacent(s: str, k: int) -> str:
    st = []                                    # [char, count]
    for ch in s:
        if st and st[-1][0] == ch:
            st[-1][1] += 1
        else:
            st.append([ch, 1])
        if st[-1][1] == k:
            st.pop()
    return "".join(ch * cnt for ch, cnt in st)


print(remove_k_adjacent("deeedbbcccbdaa", 3))   # "aa"
print(remove_k_adjacent("abbaca", 2))           # "ca"

aa
ca


# 15. Template tư duy

```python
st = []                                  # mỗi phần tử: [key..., count]
for item in sequence:
    key = <khóa của item>
    if st and <khóa đỉnh> == key:
        st[-1][count_index] += 1         # nối vào khối đỉnh
    else:
        st.append([key..., 1])           # mở khối mới
    if st[-1][count_index] == <kích thước nhóm của key>:
        st.pop()                         # đủ nhóm -> xoá, khối dưới lộ ra
# st rỗng  <=>  xoá được hết
```

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Không chạy code. Lập bảng stack từng bước cho:

```python
types     = [1, 0, 0, 1, 1, 0]
colors    = [2, 3, 3, 2, 2, 3]
groupSize = [2, 3]
```

Kết quả `True` hay `False`? Kiểm tra bằng cell dưới.

In [20]:
# Kiểm tra Bài 1 sau khi đã tính tay
trace_stack([1, 0, 0, 1, 1, 0], [2, 3, 3, 2, 2, 3], [2, 3]);

i = 0: đọc (1,2) | mở khối mới                              | stack = [[1, 2, 1]]
i = 1: đọc (0,3) | mở khối mới                              | stack = [[1, 2, 1], [0, 3, 1]]
i = 2: đọc (0,3) | nối vào đỉnh, count = 2 -> đủ 2 -> XOÁ khối | stack = [[1, 2, 1]]
i = 3: đọc (1,2) | nối vào đỉnh, count = 2                  | stack = [[1, 2, 2]]
i = 4: đọc (1,2) | nối vào đỉnh, count = 3 -> đủ 3 -> XOÁ khối | stack = []
i = 5: đọc (0,3) | mở khối mới                              | stack = [[0, 3, 1]]
Kết quả: False


## Bài 2 — Dự đoán output

Đoán từng dòng trước khi chạy:

```python
st = [[0, 1, 1], [2, 5, 2]]
print(st[-1])
st[-1][2] += 1
print(st)
st.pop()
print(st, not st)
```

In [21]:
st = [[0, 1, 1], [2, 5, 2]]
print(st[-1])
st[-1][2] += 1
print(st)
st.pop()
print(st, not st)

[2, 5, 2]
[[0, 1, 1], [2, 5, 3]]
[[0, 1, 1]] False


## Bài 3 — Hoàn thiện lời giải stack

Điền các chỗ `TODO`.

In [22]:
def exercise_3_can_remove(types: List[int], colors: List[int], groupSize: List[int]) -> bool:
    st = []
    for t, c in zip(types, colors):
        # TODO 1: nếu đỉnh stack cùng (t, c) thì tăng count, ngược lại đẩy khối mới
        # TODO 2: nếu count của đỉnh bằng groupSize[t] thì pop
        pass
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_can_remove([0, 1, 1, 0], [5, 3, 3, 5], [2, 2]) == True
# assert exercise_3_can_remove([2, 0, 1, 2], [7, 5, 4, 7], [3, 1, 2]) == False

## Bài 4 — Remove All Adjacent Duplicates (LeetCode 1047)

Cho chuỗi `s`, liên tục xoá hai ký tự **giống nhau liền kề** cho tới khi không xoá được nữa. Trả về chuỗi còn lại.

```python
remove_pairs("abbaca")   -> "ca"
remove_pairs("azxxzy")   -> "ay"
```

Thử viết **không** dùng `count` (gợi ý: khi `k = 2`, chỉ cần so với ký tự trên đỉnh).

In [23]:
def exercise_4_remove_pairs(s: str) -> str:
    # TODO
    raise NotImplementedError


# assert exercise_4_remove_pairs("abbaca") == "ca"
# assert exercise_4_remove_pairs("azxxzy") == "ay"

## Bài 5 — Đếm số thao tác xoá

Viết hàm trả về **số thao tác xoá** thực hiện được (theo thứ tự stack). Theo mục 4, con số này không phụ thuộc thứ tự xoá.

```python
count_operations([0, 1, 1, 0], [5, 3, 3, 5], [2, 2])                 -> 2
count_operations([2, 0, 1, 2], [7, 5, 4, 7], [3, 1, 2])              -> 1
count_operations([0, 0, 0, 0, 0], [1, 1, 1, 1, 1], [2])              -> 2
```

In [24]:
def exercise_5_count_operations(types: List[int], colors: List[int], groupSize: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_count_operations([0, 1, 1, 0], [5, 3, 3, 5], [2, 2]) == 2
# assert exercise_5_count_operations([2, 0, 1, 2], [7, 5, 4, 7], [3, 1, 2]) == 1
# assert exercise_5_count_operations([0, 0, 0, 0, 0], [1, 1, 1, 1, 1], [2]) == 2

## Bài 6 — Mini challenge: đếm số đoạn đầu (prefix) xoá được hết

Đếm có bao nhiêu `k` (`1 <= k <= n`) sao cho **riêng đoạn đầu** gồm `k` phần tử đầu tiên có thể xoá hết.

```python
count_removable_prefixes([0, 0, 1, 1, 0, 0], [1, 1, 2, 2, 1, 1], [2, 2])   -> 3   # k = 2, 4, 6
count_removable_prefixes([0, 1, 0], [9, 9, 9], [3, 2])                     -> 0
```

Yêu cầu: `O(n)` — không được chạy lại thuật toán cho từng `k`.

In [25]:
def exercise_6_count_removable_prefixes(types: List[int], colors: List[int], groupSize: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_6_count_removable_prefixes([0, 0, 1, 1, 0, 0], [1, 1, 2, 2, 1, 1], [2, 2]) == 3
# assert exercise_6_count_removable_prefixes([0, 1, 0], [9, 9, 9], [3, 2]) == 0

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: chỉ cần nhìn khối ở đỉnh `st[-1]`.
- Mức 2: `if st and st[-1][0] == t and st[-1][1] == c: st[-1][2] += 1 else: st.append([t, c, 1])`.
- Mức 3: `if st[-1][2] == groupSize[t]: st.pop()`; cuối cùng `return not st`.

**Bài 4**

- Mức 1: khi `k = 2`, "đủ nhóm" nghĩa là ký tự mới **trùng** ký tự trên đỉnh.
- Mức 2: trùng thì `pop`, không trùng thì `append`.
- Mức 3: `return "".join(st)`.

**Bài 5**

- Mức 1: mỗi lần `pop` là một thao tác xoá.
- Mức 2: thêm biến đếm `ops` và tăng ngay khi `pop`.
- Mức 3: dải 5 phần tử với nhóm 2 cho 2 lần xoá, dư 1.

**Bài 6**

- Mức 1: stack sau khi đọc `k` phần tử chính là trạng thái cuối của đoạn đầu dài `k` (bất biến ở mục 9).
- Mức 2: đoạn đầu dài `k` xoá hết được khi và chỉ khi stack rỗng **ngay sau** khi đọc phần tử thứ `k`.
- Mức 3: đếm số lần `not st` đúng sau mỗi bước của vòng lặp.

# 17. Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

## Đáp án Bài 1

`keys = (1,2) (0,3) (0,3) (1,2) (1,2) (0,3)`, `groupSize = [2, 3]`:

| i | đọc | hành động | stack sau |
|---:|---|---|---|
| 0 | (1,2) | mở khối | `[[1,2,1]]` |
| 1 | (0,3) | mở khối | `[[1,2,1], [0,3,1]]` |
| 2 | (0,3) | nối, count = 2 = groupSize[0] → xoá | `[[1,2,1]]` |
| 3 | (1,2) | nối vào khối vừa lộ ra, count = 2 (< 3) | `[[1,2,2]]` |
| 4 | (1,2) | nối, count = 3 = groupSize[1] → xoá | `[]` |
| 5 | (0,3) | mở khối | `[[0,3,1]]` |

Còn lại một khối `(0,3)` → `False`.

Để ý bước 3: sau khi khối `(0,3)` bị xoá, khối `(1,2)` ở đáy lộ ra và nối được với phần tử mới — hiệu ứng dồn lại. Và đoạn đầu dài 5 phần tử thì xoá hết được (stack rỗng sau bước 4), chỉ phần tử cuối là bị kẹt.

## Đáp án Bài 2

```text
[2, 5, 2]
[[0, 1, 1], [2, 5, 3]]
[[0, 1, 1]] False
```

In [26]:
# Đáp án Bài 3
def answer_3_can_remove(types: List[int], colors: List[int], groupSize: List[int]) -> bool:
    st = []
    for t, c in zip(types, colors):
        if st and st[-1][0] == t and st[-1][1] == c:
            st[-1][2] += 1
        else:
            st.append([t, c, 1])
        if st[-1][2] == groupSize[t]:
            st.pop()
    return not st


assert answer_3_can_remove([0, 1, 1, 0], [5, 3, 3, 5], [2, 2]) == True
assert answer_3_can_remove([2, 0, 1, 2], [7, 5, 4, 7], [3, 1, 2]) == False
assert answer_3_can_remove([1, 0, 0, 1, 1, 0], [2, 3, 3, 2, 2, 3], [2, 3]) == False
print("Bài 3: PASS")

Bài 3: PASS


In [27]:
# Đáp án Bài 4
def answer_4_remove_pairs(s: str) -> str:
    st = []
    for ch in s:
        if st and st[-1] == ch:
            st.pop()
        else:
            st.append(ch)
    return "".join(st)


assert answer_4_remove_pairs("abbaca") == "ca"
assert answer_4_remove_pairs("azxxzy") == "ay"
print("Bài 4: PASS")

Bài 4: PASS


In [28]:
# Đáp án Bài 5
def answer_5_count_operations(types: List[int], colors: List[int], groupSize: List[int]) -> int:
    st = []
    ops = 0
    for t, c in zip(types, colors):
        if st and st[-1][0] == t and st[-1][1] == c:
            st[-1][2] += 1
        else:
            st.append([t, c, 1])
        if st[-1][2] == groupSize[t]:
            st.pop()
            ops += 1
    return ops


assert answer_5_count_operations([0, 1, 1, 0], [5, 3, 3, 5], [2, 2]) == 2
assert answer_5_count_operations([2, 0, 1, 2], [7, 5, 4, 7], [3, 1, 2]) == 1
assert answer_5_count_operations([0, 0, 0, 0, 0], [1, 1, 1, 1, 1], [2]) == 2
print("Bài 5: PASS")

Bài 5: PASS


In [29]:
# Đáp án Bài 6
def answer_6_count_removable_prefixes(types: List[int], colors: List[int], groupSize: List[int]) -> int:
    st = []
    count = 0
    for t, c in zip(types, colors):
        if st and st[-1][0] == t and st[-1][1] == c:
            st[-1][2] += 1
        else:
            st.append([t, c, 1])
        if st[-1][2] == groupSize[t]:
            st.pop()
        if not st:                      # đoạn đầu vừa đọc xoá hết được
            count += 1
    return count


assert answer_6_count_removable_prefixes([0, 0, 1, 1, 0, 0], [1, 1, 2, 2, 1, 1], [2, 2]) == 3
assert answer_6_count_removable_prefixes([0, 1, 0], [9, 9, 9], [3, 2]) == 0
# đối chiếu với cách chạy lại cho từng prefix
random.seed(7)
for _ in range(300):
    m = random.randint(1, 3)
    gs = [random.randint(1, 3) for _ in range(m)]
    n = random.randint(1, 10)
    ty = [random.randrange(m) for _ in range(n)]
    co = [random.randint(1, 2) for _ in range(n)]
    slow = sum(can_remove_stack(ty[:k], co[:k], gs) for k in range(1, n + 1))
    assert answer_6_count_removable_prefixes(ty, co, gs) == slow
print("Bài 6: PASS")

Bài 6: PASS


# 18. Tổng kết

Bài **Remove Elements by Type and Color** kết hợp hai ý tưởng:

1. **Thứ tự xoá không quan trọng** — vì hai thao tác hoặc độc lập, hoặc chồng lên nhau trong cùng một dải giống hệt nhau (cho cùng kết quả). Nhờ vậy ta được phép **xoá tham lam**.
2. **Stack lưu khối `[type, color, count]`** — đọc từ trái sang phải, nối vào khối đỉnh, đủ nhóm thì `pop`; khối bên dưới lộ ra tự động mô phỏng việc "dồn lại".

## Công thức nhớ nhanh

```text
cùng khóa với đỉnh  -> count += 1
khác khóa           -> đẩy khối mới
count == groupSize  -> pop
stack rỗng          -> True
```

## Phiên bản nên nhớ

```python
class Solution:
    def canRemoveAll(self, types: List[int], colors: List[int], groupSize: List[int]) -> bool:
        st = []
        for t, c in zip(types, colors):
            if st and st[-1][0] == t and st[-1][1] == c:
                st[-1][2] += 1
            else:
                st.append([t, c, 1])
            if st[-1][2] == groupSize[t]:
                st.pop()
        return not st
```

## Độ phức tạp

```text
Time:  O(n)
Space: O(n)
```

# 19. Checklist trước khi submit

- [ ] Hai phần tử chỉ ghép nhóm khi **cùng loại VÀ cùng màu**?
- [ ] Kích thước nhóm lấy theo **loại**: `groupSize[t]`?
- [ ] Phần tử mới cùng khóa với đỉnh thì **nối vào** (tăng `count`), không mở khối mới?
- [ ] `pop` ngay khi `count == groupSize[t]` để khối bên dưới lộ ra?
- [ ] Đã thử test có **dây chuyền** (xoá ở giữa làm hai bên dồn lại) và dải **dài hơn** kích thước nhóm?
- [ ] Đã thử `groupSize = 1` (mọi phần tử tự xoá được) và một phần tử đơn lẻ?
- [ ] Không dùng mô phỏng `O(n²)` hay vét cạn cho `n = 10^5`?